# QA Synthesis Notebook

Interactive notebook for generating QA pairs using the Data Synthesizer pipeline.

In [ ]:
# Setup
import sys
sys.path.insert(0, '..')

%load_ext autoreload
%autoreload 2

In [ ]:
# Imports
import os
from src.core.config import load_config, _create_default_config
from src.core.logger import setup_root_logger
from src.synthesizers.qa_synthesis import QASynthesizer
from src.synthesizers.deep_thinking import DeepThinkingSynthesizer
from src.synthesizers.corpus_generator import CorpusGenerator

## Configuration

Set up your API keys and configuration.

In [ ]:
# Load configuration
config = load_config('../config.yaml')

# Or create default config
# config = _create_default_config()

# Set API key (or use environment variable)
# config.providers['gemini'].api_key = 'your-api-key'

# Configure output
config.output.type = 'local'  # 'local', 'huggingface', or 'both'
config.output.local_path = '../output'

# Set synthesis parameters
config.synthesis.num_variants = 3
config.synthesis.questions_per_topic = 5

# Domain configuration
config.domain = {
    'domain': 'Indonesian Legal System',
    'topics': [
        'Hukum Pidana dan Sanksi',
        'Hukum Perdata dan Kontrak',
        'Hukum Tata Negara'
    ]
}

print(f"Provider: {config.primary_provider}")
print(f"Topics: {len(config.domain['topics'])}")
print(f"Output: {config.output.type}")

## Initialize Synthesizer

In [ ]:
# Create synthesizer
synthesizer = QASynthesizer(config)

# Show initial progress
print("\nInitial Progress:")
synthesizer.show_progress()

## Test Single Generation

In [ ]:
# Test with a single item
test_item = {'topic': config.domain['topics'][0]}
test_id = 'test_001'

results = synthesizer.synthesize_item(test_item, test_id)

print(f"Generated {len(results)} variants:\n")
for r in results:
    print(f"Q: {r['question'][:80]}...")
    print(f"A: {r['answer'][:100]}...")
    print()

## Run Full Synthesis

In [ ]:
# Run synthesis for all topics
# This will continue from where it left off
synthesizer.run()

## Check Progress

In [ ]:
# Show final progress
synthesizer.show_progress()

# Get usage stats
usage = synthesizer.provider.get_usage()
print(f"\nAPI Usage: {usage['total_requests']} requests, {usage['total_tokens']} tokens")

## Deep Thinking Synthesis

Generate QA pairs with detailed reasoning traces.

In [ ]:
# Configure for deep thinking
config.synthesis.type = 'deep_thinking'
config.synthesis.num_variants = 2
config.domain['min_thinking_words'] = 300
config.domain['max_answer_words'] = 150

# Create deep thinking synthesizer
deep_synth = DeepThinkingSynthesizer(config)

# Test generation
results = deep_synth.synthesize_item({'topic': 'Legal Case Analysis'}, 'deep_test_001')

if results:
    r = results[0]
    print(f"Question: {r['question']}\n")
    print(f"Thinking ({r['thinking_word_count']} words):")
    print(r['thinking'][:500] + '...\n')
    print(f"Answer ({r['answer_word_count']} words):")
    print(r['answer'])

## Corpus Generation

Generate full documents like contracts.

In [ ]:
# Configure for corpus generation
config.synthesis.type = 'corpus'
config.domain = {
    'domain': 'Legal Contracts',
    'min_document_length': 500,
    'taxonomy': {
        'document_types': ['NDA', 'Service Agreement'],
        'industries': ['Technology'],
        'jurisdictions': ['US Law']
    }
}

# Create corpus generator
corpus_gen = CorpusGenerator(config)

# Test generation
results = corpus_gen.synthesize_item({'topic': 'NDA'}, 'corpus_test_001')

if results:
    r = results[0]
    print(f"Document Type: {r['document_type']}")
    print(f"Word Count: {r['word_count']}")
    print(f"\nDocument Preview:")
    print(r['document_text'][:1000] + '...')